## 画像データ向けの大規模な TFRecord の作成
このラボでは、画像分類データセット用の TFRecord を作成し、画像データ用の TFRecord を作成する Apache Beam パイプラインを開発して、これらの TFRecord を大規模に作成する Dataflow ジョブを送信します。

学習目標:
* 画像データを大規模に TFRecord にシリアライズする方法を学ぶ
* `tf.Example` プロトコルバッファを作成するために `beam.DoFn` を拡張する方法を学ぶ
* Beam パイプラインから TFRecord を書き出すために `beam.io.tfrecordio.WriteToTFRecord` を使用する方法を学ぶ
* Apache Beam パイプラインを Dataflow にデプロイする方法を学ぶ

#### セットアップ

In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = PROJECT + "-flowers"
REGION = "us-central1"
CLASSES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]
DATASET_FILE = f"gs://{BUCKET}/flowers.csv"

%env PROJECT={PROJECT}
%env BUCKET={BUCKET}
%env REGION={REGION}
%env DATASET_FILE={DATASET_FILE}

データの保存には GCS を使用します。次のセルを実行してこのバケットを作成します。

In [ ]:
%%bash
exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
  echo -e "Bucket gs://${BUCKET} already exists."
    
else
   echo "Creating a new GCS bucket."
   gcloud storage buckets create --location=${REGION} gs://${BUCKET}
   echo -e "\nHere are your current buckets:"
   gcloud storage ls
fi

データセットの CSV ファイルを GCS バケットにコピーします。

In [ ]:
!gcloud storage cp ../dataset_import_files/flowers.csv {DATASET_FILE}

#### データセットのインポートファイルの確認

In [ ]:
examples = pd.read_csv(DATASET_FILE)
examples.columns = ["imageUri", "label"]
examples.head()

ご覧のように、この画像分類データセットは `Image URI, Label` の形式を持つ CSV ファイルとして表現されています。画像 URI は、一般公開されている GCS バケットに保存された JPEG 画像を指しています。このファイルは、データの `.tfrecord` ファイルを作成する Beam パイプラインへの入力として使用されます。

#### 1 つのサンプルの確認

In [ ]:
single_example = examples.iloc[0]
img = tf.io.decode_jpeg(tf.io.read_file(single_example.imageUri))
plt.imshow(img)
plt.title(single_example.label)
plt.show()

### Apache Beam と Dataflow を使用した大規模な TFRecord の作成
画像データでは、非常に大規模なデータセットを扱うのが一般的です。このような状況では、ローカルで画像を TFRecord に変換することは現実的ではありません。Cloud Dataflow の大規模なスケーラビリティを活用して、この処理を代行させることができます。

Dataflow は Apache Beam の実行エンジンであるため、CSV ファイルから読み込み、データセット全体を TFRecord にシリアライズする Beam パイプラインを作成する必要があります。

#### Apache Beam パイプライン
この Apache Beam パイプラインは次の処理を行います:
* `imageURI, label` 形式の CSV ファイルから行を読み込む
* カスタム `beam.DoFn` で TF Example を作成する
* `beam.Partition` を使用してデータをトレーニング用と検証用にランダムに分割する
* Example を TFRecord にシリアライズする
* `beam.io.tfrecordio.WriteToTFRecord` を使用してデータを GCS に書き出す

パイプラインのコードはすべて同じファイルに書き込まれますが、わかりやすくするために複数のセルに分割して記述します。

まず、インポートを定義します。

In [ ]:
!mkdir -p create_tfrecords

In [ ]:
%%writefile create_tfrecords/create_tfrecords.py
import apache_beam as beam
from apache_beam.options.pipeline_options import (
    GoogleCloudOptions, 
    PipelineOptions, 
    StandardOptions, 
    SetupOptions
)

from apache_beam.runners import DataflowRunner, DirectRunner
import argparse
import random 

import tensorflow as tf
import typing

データセット内の各サンプルの画像 URI とラベルの `NamedTuple` を返すように、`str.split()` メソッドを使用して各行を分割する `ParseCsv`（カスタム `DoFn`）を定義します。

In [ ]:
%%writefile -a create_tfrecords/create_tfrecords.py

# CSV ファイルのスキーマ
class CSVRow(typing.NamedTuple):
    image_uri: str
    label: str

# CSV 行をスキーマ付き PCollection に変換する DoFn
class ParseCsv(beam.DoFn):
    def process(self, element):
        image_uri, label = element.split(',')
        yield CSVRow(
            image_uri = image_uri,
            label = label
        )


画像データを TF Example にシリアライズする `CreateTFExample` とヘルパー関数を定義します。また、トレーニング／テスト分割用にデータをランダムに分割する関数を定義します。注: ランダムな分割は再現性がありません。

In [ ]:
%%writefile -a create_tfrecords/create_tfrecords.py

# TFRecord ヘルパー関数
def _int64_feature(value):
    """bool / enum / int / uint から int64_list を返します。"""
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def _image_feature(value):
    """画像から bytes_list を返します。"""
    return tf.train.Feature(
        bytes_list=tf.train.BytesList(value=[tf.io.encode_jpeg(value).numpy()])
    )

# TF Example を作成する DoFn
class CreateTFExample(beam.DoFn):
    def process(self, element):
        CLASSES = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
        img = tf.io.decode_jpeg(tf.io.read_file(element.image_uri))
        
        feature = {
            "image": _image_feature(img), 
            "label": _int64_feature(CLASSES.index(element.label)),
        }
        
        yield tf.train.Example(features=tf.train.Features(feature=feature))
        
def partition_fn(example, num_partitions, train_percent):
    if random.random() < train_percent:
        return 0
    return 1


`run` 関数を定義します。この関数はコマンドライン引数を解析し、パイプラインオプションを設定してから、Beam パイプライン自体を構成します。

In [ ]:
%%writefile -a create_tfrecords/create_tfrecords.py
    
# Beam パイプラインを実行する関数
def run():
    parser = argparse.ArgumentParser(description='Image Data to TFRecords')

    # Google Cloud オプション
    parser.add_argument('--project',required=True, help='Specify Google Cloud project')
    parser.add_argument('--region', required=True, help='Specify Google Cloud region')
    parser.add_argument('--staging_location', required=True, help='Specify Cloud Storage bucket for staging')
    parser.add_argument('--runner', required=True, help='Specify Apache Beam Runner')
    parser.add_argument('--job_name', required=True, help='Job name for Dataflow Runner')

    # パイプライン固有のオプション
    parser.add_argument('--dataset_file', required=True, help='GCS path to input CSV')
    parser.add_argument('--output_dir', required=True, help='GCS output directory')
    parser.add_argument('--train_percent', required=True, help='Percentage of training data')
    parser.add_argument('--requirements_file', required=True, help='Required Packages')


    opts, pipeline_opts = parser.parse_known_args()

    # Beam パイプラインオプションの設定
    options = PipelineOptions(pipeline_opts)

    DATASET_FILE = opts.dataset_file
    OUTPUT_DIR = opts.output_dir
    TRAIN_PERCENT = float(opts.train_percent)
    
    # 標準パイプラインオプションの設定
    options.view_as(StandardOptions).streaming = False
    options.view_as(StandardOptions).runner = opts.runner
    options.view_as(SetupOptions).save_main_session = True
    options.view_as(SetupOptions).requirements_file = opts.requirements_file
    options.view_as(SetupOptions).pickle_library = "dill"

    # Google Cloud 固有オプションの設定
    google_cloud_options = options.view_as(GoogleCloudOptions)
    google_cloud_options.project = opts.project
    google_cloud_options.job_name = opts.job_name
    google_cloud_options.staging_location = opts.staging_location
    google_cloud_options.region = opts.region


    # パイプラインのインスタンス化
    if opts.runner=='DataflowRunner':
        p = beam.Pipeline(DataflowRunner(), options=options)
    else:
        p = beam.Pipeline(DirectRunner(), options=options)
        
    rows = ( p | "Read CSV" >> beam.io.ReadFromText(DATASET_FILE)
               | "Parse CSV" >> beam.ParDo(ParseCsv()))
    
    train, val = ( rows 
                   | "Create TF Examples" >> beam.ParDo(CreateTFExample()) 
                   | "Split Data" >> beam.Partition(partition_fn, 2, train_percent=TRAIN_PERCENT))
    
    write_train = ( train 
                       | "Serialize Training Examples" >> beam.Map(lambda x: x.SerializeToString()) 
                       | "Write Train" >> beam.io.tfrecordio.WriteToTFRecord(
                           f"{OUTPUT_DIR}/train.tfrecord", num_shards=10
                       )
                  )
    write_val = ( val 
                       | "Serialize Validation Examples" >> beam.Map(lambda x: x.SerializeToString()) 
                       | "Write Validation" >> beam.io.tfrecordio.WriteToTFRecord(
                           f"{OUTPUT_DIR}/eval.tfrecord", num_shards=3
                       )
                )
    
    # パイプラインの実行
    p.run()
    
if __name__ == '__main__':
    run()


### このパイプラインの詳細

#### Apache Beam の中核概念
`Pcollection`: データ要素を表す不変（immutable）な値のコレクションです。

`PTransform`: パイプラインにおけるデータ処理操作（またはステップ）を表します。

`ParDo`: 汎用的な並列処理のための変換（transform）です。`ParDo` 変換は、入力 `PCollection` の各要素を受け取り、その要素に対して何らかの処理関数を実行して、出力 `PCollection` に要素を送出します。ParDo に渡される処理関数は `DoFn` オブジェクトです。

`DoFn`: パイプラインの具体的なデータ処理タスクを定義するものです。カスタム処理タスクを作成するには、`DoFn` のサブクラス（例: `class MyCustomProcessingTask(beam.DoFn)`）を作成し、実際の処理ロジックを提供する `def process(self, element):` メソッドを記述します。入力コレクションから要素を手動で取り出す必要はありません。Beam SDK がそれを代行します。`process` メソッドは入力要素である引数 `element` を受け取り、その出力値を持つ反復可能オブジェクト（iterable）を返す必要があります。これは `yield` 文で個々の要素を送出することで実現できます。

#### 画像データから TFRecord へのパイプライン
1) `beam.io.ReadFromText`: テキストファイルを読み込んで `str` 要素にする `PTransform` です。ファイル内の各行に対して 1 つの要素を返します。今回の画像データセットの入力 CSV ファイルでは、CSV の各行に対して `"{imageUri}, {label}"` という文字列要素を返します。

2) `beam.ParDo(ParseCsv())`: `beam.io.ReadFromText` からの出力 `PCollection` 内の各文字列要素に `ParseCsv`（カスタム `DoFn`）を適用する `PTransform` です。`ParseCsv` の `process` メソッドは、単純に `str.split()` メソッドを使用して各行を分割し、データセット内の各サンプルの画像 URI とラベルの `NamedTuple` を返します。

3) `beam.ParDo(CreateTFExample())`: `beam.ParDo(ParseCsv())` からの出力 `PCollection` 内の各要素に `CreateTFExample()`（カスタム `DoFn`）を適用する `PTransform` です。`CreateTFExample()` の `process` メソッドは、まず `tf.io.read_file` と `tf.io.decode_jpeg` を使用して画像 URI から画像を読み込んでデコードし、TFRecord ヘルパー関数で特徴量辞書を作成して、`tf.train.Example` プロトコルバッファを返します。

4) `beam.Partition(partition_fn, 2, train_percent)`: コレクション内の要素を複数の出力コレクションに分離する `PTransform` です。提供された分割関数には、要素を各結果パーティションにどのように分離するかを決定するロジックが含まれています。今回のパイプラインでは、カスタム `partition_fn` により、乱数が train_percent 未満の場合はトレーニングパーティションに、乱数が train_percent 以上の場合はテスト（検証）パーティションに入力コレクションを 2 つに分割します。

パイプラインのこの段階で、`train` と `val` という 2 つの `PCollection` が得られます。この時点で、`train` と `val` の各要素は `tf.train.Example` プロトコルバッファになっています。これら両方について、シリアライズして `.tfrecord` 形式で書き出す必要があります。これら各 `PCollection` を次の 2 つの `PTransform` にパイプします:
1) `beam.Map(lambda x: x.SerializeToString())`: `PCollection` 内の各 `tf.train.Example` プロトコルバッファ要素に `.SerializeToString` を適用する `PTransform` です。

2) `beam.io.tfrecordio.TFRecord(OUTPUT_DIR)`: 指定された出力ディレクトリに、シリアライズされた `tf.train.Example` プロトコルバッファを `.tfrecord` 形式で書き出す `PTransform` です。

### Dataflow 用の requirements.txt ファイルの作成

In [ ]:
%%writefile create_tfrecords/requirements.txt
tensorflow==2.18.1

### Cloud Dataflow によるパイプラインの実行

In [ ]:
TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
JOB_NAME = f"images-to-tfrecords-{TIMESTAMP}"
OUTPUT_DIR = f"gs://{BUCKET}/data"
STAGING_LOCATION = f"gs://{BUCKET}/staging"
TRAIN_PERCENT = 0.8

%env JOB_NAME = {JOB_NAME}
%env OUTPUT_DIR = {OUTPUT_DIR}
%env STAGING_LOCATION = {STAGING_LOCATION}
%env TRAIN_PERCENT = {TRAIN_PERCENT}

In [ ]:
%%bash
python3 create_tfrecords/create_tfrecords.py \
    --project=${PROJECT} \
    --region=${REGION} \
    --runner='DataflowRunner' \
    --job_name=${JOB_NAME} \
    --staging_location=${STAGING_LOCATION} \
    --dataset_file=${DATASET_FILE} \
    --output_dir=${OUTPUT_DIR} \
    --train_percent=${TRAIN_PERCENT} \
    --requirements_file='./create_tfrecords/requirements.txt'

**注**: Dataflow ジョブの完了には約 10 分かかります。Dataflow UI でジョブの進行状況を確認しても構いません。
先に進む前にジョブが完了している必要があります。

これで、データセット全体が TFRecord にシリアライズされ、GCS に保存されました。作成された TFRecord を確認するには、次のコマンドを実行します。`CommandException: One of more URLs matched no objects` というエラーが表示された場合は、Dataflow ジョブがまだ終了していない可能性があります。

In [ ]:
!gcloud storage ls -l {OUTPUT_DIR}

`tf.data` API を使用して、いくつかのサンプルを取り込み、解析して表示します。

In [ ]:
FILE_PATTERN = OUTPUT_DIR + "/train*"


def parse_example(example):
    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.int64),
    }
    example = tf.io.parse_single_example(example, feature_description)
    example["image"] = tf.io.decode_jpeg(example["image"], channels=3)
    return example


ds = tf.data.TFRecordDataset(tf.io.gfile.glob(FILE_PATTERN))
ds = ds.map(parse_example)

for example in ds.take(2):
    plt.imshow(example["image"].numpy())
    plt.title(CLASSES[example["label"].numpy()])
    plt.show()

お疲れさまでした。画像データセット全体を大規模に TFRecord へシリアライズする Dataflow パイプラインを正常にデプロイできました。

# License
Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.